## Function: Setup PostGIS + OSM Workflow 🗄️

Build a reusable function to create a database, enable PostGIS, download OSM data, and load it.

### ⚙️ Step 0: Select Kernel
Use **python-gis-development (.venv)**

### 📚 Step 1: Imports

In [ ]:
import os
import requests
import subprocess
import psycopg2

### 🔌 Step 2: Connect to PostgreSQL

In [ ]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)
conn.autocommit = True
cur = conn.cursor()
print("Connected")

### 🗄️ Step 3: Create Database

In [ ]:
cur.execute("CREATE DATABASE osm_db;")
cur.close()
conn.close()
print("Database created")

### 🌍 Step 4: Enable PostGIS

In [ ]:
conn = psycopg2.connect(
    dbname="osm_db",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("CREATE EXTENSION postgis;")

cur.close()
conn.close()
print("PostGIS enabled")

### ⬇️ Step 5: Download OSM

In [ ]:
osm_url = "https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"
data_path = "data/osm"
os.makedirs(data_path, exist_ok=True)

file_path = os.path.join(data_path, osm_url.split("/")[-1])

if not os.path.exists(file_path):
    r = requests.get(osm_url, stream=True)
    with open(file_path, "wb") as f:
        for chunk in r.iter_content(1024):
            f.write(chunk)

print("Downloaded")

### 📥 Step 6: Load with osm2pgsql

In [ ]:
cmd = [
    "osm2pgsql",
    "-d", "osm_db",
    "-U", "postgres",
    "--create",
    "--slim",
    file_path
]

subprocess.run(cmd, check=True)
print("Loaded into PostGIS")

### 🧩 Step 7: Build Function

In [ ]:
def setup_postgis_osm(db_name="osm_db", osm_url=None):
    import os, requests, subprocess, psycopg2

    data_path = "data/osm"
    os.makedirs(data_path, exist_ok=True)

    file_path = os.path.join(data_path, osm_url.split("/")[-1])

    if not os.path.exists(file_path):
        r = requests.get(osm_url, stream=True)
        with open(file_path, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

    conn = psycopg2.connect(dbname="postgres", user="postgres", password="postgres", host="localhost", port=5432)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE {db_name};")
    cur.close()
    conn.close()

    conn = psycopg2.connect(dbname=db_name, user="postgres", password="postgres", host="localhost", port=5432)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION postgis;")
    cur.close()
    conn.close()

    subprocess.run([
        "osm2pgsql",
        "-d", db_name,
        "-U", "postgres",
        "--create",
        "--slim",
        file_path
    ], check=True)

    return True

### 🧪 Step 8: Test

In [ ]:
setup_postgis_osm(
    osm_url="https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"
)
print("Done")